# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (BF16)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out no cells below after first installation. Colab has fresh env everytime. UV parallelizes installations so should be quick.

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!wget -qO- https://astral.sh/uv/install.sh | sh

# Make uv findable in subsequent cells
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

# Verify
!which uv && uv --version

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/usr/local/bin/uv
uv 0.11.8 (x86_64-unknown-linux-gnu)


In [3]:
constraints = """torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57
"""

with open("/content/constraints.txt", "w") as f:
    f.write(constraints)

!cat /content/constraints.txt

torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57


In [4]:
!uv pip install --system torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 96ms


In [5]:
import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("CUDA version torch was built with:", torch.version.cuda)

torch version: 2.5.1+cu121
CUDA available: True
Device count: 1
Device name: NVIDIA A100-SXM4-80GB
CUDA version torch was built with: 12.1


In [6]:
# removed bitsandbytes package cause not using it anymore.
!uv pip install --system \
    sympy numpy transformers vllm tqdm \
    antlr4-python3-runtime==4.11.1 accelerate \
    -c /content/constraints.txt

Using Python 3.12.13 environment at: /usr
Checked 7 packages in 102ms


In [7]:
# If vllm import fails the first time after install, restart the session and re-run all cells. Same for any persistent setup errors.
import torch
import vllm
print("Post-restart checks:")
print("  torch:", torch.__version__)
print("  CUDA available:", torch.cuda.is_available())
print("  Device:", torch.cuda.get_device_name(0))
print("  vllm:", vllm.__version__)
print("  GPU memory free:", round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), "GB")

Post-restart checks:
  torch: 2.5.1+cu121
  CUDA available: True
  Device: NVIDIA A100-SXM4-80GB
  vllm: 0.7.3
  GPU memory free: 84.65 GB


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_DIR` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [8]:
from pathlib import Path
import sys
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/cse151b")
GIVEN_DATA_DIR = PROJECT_DIR / "given_data"
OUTPUT_DIR = PROJECT_DIR  # final outputs land here on Drive

# Local hot-path directory (Colab disk — fast, reliable, but wiped on runtime death)
LOCAL_DIR = Path("/content/local_results")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Inputs
# DATA_PATH = str(GIVEN_DATA_DIR / "public.jsonl")
DATA_PATH = str(GIVEN_DATA_DIR / "private.jsonl")

# Hot-path outputs — written to local disk during generation
RESPONSES_PATH = LOCAL_DIR / "responses.jsonl"
LOG_PATH = LOCAL_DIR / "responses.log"

# Drive backup of responses (snapshot target during run, restore source after restart)
RESPONSES_BACKUP = OUTPUT_DIR / "responses.jsonl"

# Final outputs — written to Drive (only one write each, at the end of their step)
SCORED_PATH = OUTPUT_DIR / "scored_results.jsonl"
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

# Make given_data importable
sys.path.insert(0, str(GIVEN_DATA_DIR))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert OUTPUT_DIR.exists(), f"Drive not mounted? {OUTPUT_DIR} missing"

# If a Drive backup exists from a previous session but no local copy, restore it
if RESPONSES_BACKUP.exists() and not RESPONSES_PATH.exists():
    shutil.copy2(RESPONSES_BACKUP, RESPONSES_PATH)
    print(f"Restored {RESPONSES_PATH.stat().st_size} bytes from Drive backup")

print(f"Project: {PROJECT_DIR}")
print(f"Inputs:  {GIVEN_DATA_DIR}")
print(f"Hot dir: {LOCAL_DIR}")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Existing local responses: {RESPONSES_PATH.exists()}")
print(f"Existing Drive backup:    {RESPONSES_BACKUP.exists()}")

Project: /content/drive/MyDrive/cse151b
Inputs:  /content/drive/MyDrive/cse151b/given_data
Hot dir: /content/local_results
Outputs: /content/drive/MyDrive/cse151b
Existing local responses: False
Existing Drive backup:    False


In [1]:
import csv

In [ ]:
import csv
import json
import re
import time
from collections import Counter
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
MAX_TOKENS  = 32768 # was 32768. 38912 recommended

# Set GPU_ID earlier if training off of colab and have multiple GPUs. Need to declare before torch is imported.
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [10]:
with open(DATA_PATH) as f:
    data = [json.loads(line) for line in f]

In [11]:
n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 943 questions  (300 MCQ, 643 free-form)


In [12]:
# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))


── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [13]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "If the problem has multiple sub-answers, place them inside a single \\boxed{} "
    "separated by commas, e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices, then select the single best option. "
    "Please reason step by step, and put only the letter of your chosen option "
    "within \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [14]:
# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with vLLM

We load Qwen3-4B-Thinking-2507 in BF16 for maximum throughput on A100. Quantization (BnB INT8) is said to be slower (needs verification) than BF16 on A100 due to dequantization overhead exceeding memory bandwidth savings. AWQ would be the right choice if memory becomes tight.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",                   # drop bnb on A100 — BF16 is faster AND better quality
    trust_remote_code=True,
    max_model_len=24576,
    gpu_memory_utilization=0.92,
    max_num_batched_tokens=24576,       # match max_model_len; 32768 was overprovisioned
    max_num_seqs=64,                   # 256 is fine if you have headroom, 128 is safer. Tests set OOM on 128.
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
    disable_log_stats=True,
)

sampling_params = SamplingParams(
    max_tokens=24576,                   # tame the long-tail thinking traces
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    n=4,                                # Sampling_params -- Majority Vote
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed = 5,
    stop=None,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-02 06:55:59 model_runner.py:1115] Loading model weights took 7.5454 GB
INFO 05-02 06:56:01 worker.py:267] Memory profiling takes 1.91 seconds
INFO 05-02 06:56:01 worker.py:267] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.92) = 72.91GiB
INFO 05-02 06:56:01 worker.py:267] model weights take 7.55GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.83GiB; the rest of the memory reserved for KV Cache is 63.45GiB.
INFO 05-02 06:56:02 executor_base.py:111] # cuda blocks: 28876, # CPU blocks: 1820
INFO 05-02 06:56:02 executor_base.py:116] Maximum concurrency for 24576 tokens per request: 18.80x
INFO 05-02 06:56:04 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 11/11 [00:11<00:00,  1.02s/it]

INFO 05-02 06:56:15 model_runner.py:1562] Graph capturing finished in 11 secs, took 0.23 GiB
INFO 05-02 06:56:15 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 16.26 seconds


Model loaded.


In [16]:
import utils
from utils import last_boxed_only_string, remove_boxed
from judger import Judger

In [17]:
def extract_letter(text: str) -> str:
    # Strip thinking trace — only look at content after </think>
    think_end = text.rfind("</think>")
    search_text = text[think_end + len("</think>"):] if think_end >= 0 else text

    # First try: pull the last \boxed{...} content using utils' brace-aware parser
    boxed = last_boxed_only_string(search_text)
    if boxed is not None:
        inner = remove_boxed(boxed)
        if inner:
            m = re.search(r"[A-Za-z]", inner)
            if m:
                return m.group(0).upper()
    # Fallback: last standalone capital letter in the post-think response
    matches = re.findall(r"\b([A-Z])\b", search_text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

In [18]:
judger = Judger(strict_extract=False)
print("Scoring helpers ready.")

Scoring helpers ready.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [19]:
CHUNK_SIZE = 250
completed_ids = set()

In [20]:
# ── Resume: skip IDs we already have responses for ────────────────────────
if RESPONSES_PATH.exists():
    with open(RESPONSES_PATH) as f:
        for line in f:
            try:
                completed_ids.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                continue

In [21]:
remaining = [d for d in data if d.get("id") not in completed_ids]
print(f"Resuming generation: {len(completed_ids)} done, {len(remaining)} to go")

Resuming generation: 0 done, 943 to go


In [22]:
def make_prompt(item):
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

In [23]:
t_start = time.time()
pbar = tqdm(total=len(remaining), desc="Generating", unit="q")

with open(RESPONSES_PATH, "a") as fout, open(LOG_PATH, "a") as flog:
    flog.write(f"\n=== Generation started {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n")
    flog.flush()

    for i in range(0, len(remaining), CHUNK_SIZE):
        chunk = remaining[i : i + CHUNK_SIZE]
        prompts = [make_prompt(item) for item in chunk]
        t_chunk = time.time()

        try:
            outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=False)
        except Exception as e:
            flog.write(f"CHUNK FAIL items {i}-{i+len(chunk)}: {e}\n")
            flog.flush()
            # Fall back to one-by-one to isolate the bad item
            outputs = []
            for p, item in zip(prompts, chunk):
                try:
                    outputs.extend(llm.generate([p], sampling_params=sampling_params, use_tqdm=False))
                except Exception as e2:
                    flog.write(f"  ITEM FAIL id={item.get('id')}: {e2}\n")
                    outputs.append(None)
            flog.flush()

        for item, out in zip(chunk, outputs):
            if out is None:
                record = {
                    "id": item.get("id"),
                    "is_mcq": bool(item.get("options")),
                    "gold": item.get("answer"),
                    "response": "",
                    "error": "generation_failed",
                }
            else:
                record = {
                    "id": item.get("id"),
                    "is_mcq": bool(item.get("options")),
                    "gold": item.get("answer"),
                    "response": out.outputs[0].text.strip(),
                }
            fout.write(json.dumps(record) + "\n")
            pbar.update(1)

        fout.flush(); os.fsync(fout.fileno())

        chunk_time = time.time() - t_chunk
        elapsed = time.time() - t_start
        msg = (f"Chunk {i//CHUNK_SIZE+1}: {len(chunk)} items in {chunk_time:.1f}s "
               f"| elapsed: {elapsed/60:.1f}min")
        flog.write(msg + "\n"); flog.flush()

        # Snapshot to Drive every 5 chunks (~250 items)
        chunk_idx = i // CHUNK_SIZE
        if chunk_idx > 0 and chunk_idx % 5 == 0:
            try:
                import shutil
                shutil.copy2(RESPONSES_PATH, RESPONSES_BACKUP)
                flog.write(f"  Drive backup: {RESPONSES_BACKUP}\n"); flog.flush()
            except Exception as e:
                flog.write(f"  Drive backup FAILED: {e}\n"); flog.flush()
                # Don't crash the run — local file is the source of truth

pbar.close()

# Final copy to Drive so subsequent steps (and the next session) have it
import shutil
try:
    shutil.copy2(RESPONSES_PATH, RESPONSES_BACKUP)
    print(f"\nGeneration complete.")
    print(f"  Local: {RESPONSES_PATH}")
    print(f"  Drive: {RESPONSES_BACKUP}")
except Exception as e:
    print(f"\nGeneration complete locally but Drive copy failed: {e}")
    print(f"  Manually copy {RESPONSES_PATH} to Drive before relying on it.")

Generating:   0%|          | 0/943 [00:00<?, ?q/s]

WARNING 05-02 07:10:00 scheduler.py:1754] Sequence group 204 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1
WARNING 05-02 07:13:55 scheduler.py:1754] Sequence group 201 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=51
WARNING 05-02 07:17:11 scheduler.py:1754] Sequence group 222 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=101
WARNING 05-02 07:39:12 scheduler.py:1754] S

Generating:   0%|          | 1/943 [1:08:39<1077:49:32, 4119.08s/q]

WARNING 05-02 08:25:10 scheduler.py:1754] Sequence group 475 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=201


Generating:  27%|██▋       | 251/943 [2:08:56<4:59:04, 25.93s/q]   

WARNING 05-02 09:20:26 scheduler.py:1754] Sequence group 674 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=251
WARNING 05-02 09:25:11 scheduler.py:1754] Sequence group 692 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=301
WARNING 05-02 09:29:56 scheduler.py:1754] Sequence group 700 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=351
WARNING 05-02 09:33:35 scheduler.py:1754

Generating: 100%|██████████| 943/943 [4:33:13<00:00, 17.38s/q]


Generation complete.
  Local: /content/local_results/responses.jsonl
  Drive: /content/drive/MyDrive/cse151b/responses.jsonl


In [24]:
print(f"pbar.n: {pbar.n} / {pbar.total}")
print(f"len(remaining): {len(remaining)}")
print(f"len(completed_ids): {len(completed_ids)}")
with open(RESPONSES_PATH) as f:
    lines = f.readlines()
print(f"Records in file: {len(lines)}")

pbar.n: 943 / 943
len(remaining): 943
len(completed_ids): 0
Records in file: 943


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [25]:
# Load responses from Step 6
records = []
with open(RESPONSES_PATH) as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue

print(f"Loaded {len(records)} responses to score")

Loaded 943 responses to score


In [26]:
results = []

In [27]:
import signal

class ScoringTimeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise ScoringTimeout("scoring timeout")

# Register handler once, outside the loop
signal.signal(signal.SIGALRM, _timeout_handler)

<Handlers.SIG_DFL: 0>

In [28]:
for r in tqdm(records, desc="Scoring"):
    # Carry forward generation failures without crashing the scorer
    if r.get("error") == "generation_failed":
        r["correct"] = False
        results.append(r)
        continue

    is_mcq = r["is_mcq"]
    gold = r["gold"]
    response = r["response"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        signal.alarm(120)  # 120 second per-item ceiling
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False
        finally:
            signal.alarm(0)

    r["correct"] = correct
    results.append(r)

Scoring: 100%|██████████| 943/943 [21:07<00:00,  1.34s/it]  


In [29]:
# Save the scored file
with open(SCORED_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"Scoring complete. {len(results)} results written to {SCORED_PATH}")

Scoring complete. 943 results written to /content/drive/MyDrive/cse151b/scored_results.jsonl


In [14]:
with open(SCORED_PATH) as f:
    results = [json.loads(line) for line in f]

# ── Bucket failures by likely cause ────────────────────────────────────────────
def diagnose(r):
    """Return a category for why this response failed (or 'correct' if it didn't)."""
    if r.get("correct"):
        return "correct"
    if r.get("error") == "generation_failed":
        return "generation_failed"

    response = r.get("response", "")
    if not response:
        return "empty_response"

    # Check for thinking trace presence and closure
    has_think_open = "<think>" in response
    has_think_close = "</think>" in response

    # Find where the actual answer should be (post-think content)
    if has_think_close:
        post_think = response.split("</think>", 1)[1]
    else:
        post_think = response  # no think tags, treat whole thing

    # Check for boxed answer
    has_boxed = bool(re.search(r"\\boxed\{", response))
    has_boxed_post_think = bool(re.search(r"\\boxed\{", post_think))

    # Heuristic: did generation hit the token limit?
    # Qwen3-Thinking responses near MAX_TOKENS likely got truncated.
    # We don't have token counts, so use char length as a proxy (~4 chars/token).
    response_len = len(response)
    likely_truncated = response_len > 40000  # ~10k tokens, near the 12288 limit

    # Categorize
    if has_think_open and not has_think_close:
        return "truncated_in_thinking"
    if has_think_close and not has_boxed_post_think:
        if not post_think.strip():
            return "empty_after_think"
        return "no_boxed_in_answer"
    if not has_boxed:
        return "no_boxed_anywhere"
    if likely_truncated:
        return "likely_token_limit"

    # Has boxed answer but still wrong
    if r["is_mcq"]:
        return "mcq_wrong_letter"
    return "freeform_wrong_value"


categories = Counter()
by_cat_examples = {}
for r in results:
    cat = diagnose(r)
    categories[cat] += 1
    by_cat_examples.setdefault(cat, []).append(r)

# ── Print summary ─────────────────────────────────────────────────────────────
total = len(results)
n_correct = categories.get("correct", 0)
n_wrong = total - n_correct

print("=" * 60)
print(f"FAILURE ANALYSIS  ({n_wrong}/{total} incorrect = {100*n_wrong/total:.1f}%)")
print("=" * 60)

for cat, count in categories.most_common():
    pct_of_total = 100 * count / total
    pct_of_wrong = 100 * count / n_wrong if n_wrong > 0 else 0
    bar = "█" * int(pct_of_total / 2)
    print(f"  {cat:30s} {count:4d}  ({pct_of_total:5.1f}% of all, {pct_of_wrong:5.1f}% of wrong) {bar}")

# ── MCQ vs free-form breakdown ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("BY QUESTION TYPE")
print("=" * 60)

for q_type_label, q_filter in [("MCQ", lambda r: r["is_mcq"]),
                                ("Free-form", lambda r: not r["is_mcq"])]:
    subset = [r for r in results if q_filter(r)]
    correct_count = sum(1 for r in subset if r.get("correct"))
    print(f"\n{q_type_label}: {correct_count}/{len(subset)} = {100*correct_count/len(subset):.1f}%")
    sub_cats = Counter(diagnose(r) for r in subset if not r.get("correct"))
    for cat, count in sub_cats.most_common():
        print(f"  {cat:30s} {count:4d}")

# ── Response length stats ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("RESPONSE LENGTH DISTRIBUTION (chars)")
print("=" * 60)

correct_lens = [len(r.get("response", "")) for r in results if r.get("correct")]
wrong_lens = [len(r.get("response", "")) for r in results if not r.get("correct") and r.get("response")]

def stats(lst, label):
    if not lst:
        print(f"  {label}: no data")
        return
    s = sorted(lst)
    print(f"  {label:15s} n={len(s):4d}  "
          f"min={s[0]:6d}  p25={s[len(s)//4]:6d}  "
          f"median={s[len(s)//2]:6d}  p75={s[3*len(s)//4]:6d}  "
          f"max={s[-1]:6d}")

stats(correct_lens, "Correct")
stats(wrong_lens, "Wrong")

# Truncation rate by correctness
truncated_threshold = 40000
n_correct_long = sum(1 for x in correct_lens if x > truncated_threshold)
n_wrong_long = sum(1 for x in wrong_lens if x > truncated_threshold)
print(f"\n  Likely truncated (>{truncated_threshold} chars):")
print(f"    Correct: {n_correct_long}/{len(correct_lens)} ({100*n_correct_long/max(len(correct_lens),1):.1f}%)")
print(f"    Wrong:   {n_wrong_long}/{len(wrong_lens)} ({100*n_wrong_long/max(len(wrong_lens),1):.1f}%)")

FAILURE ANALYSIS  (483/1126 incorrect = 42.9%)
  correct                         643  ( 57.1% of all, 133.1% of wrong) ████████████████████████████
  freeform_wrong_value            301  ( 26.7% of all,  62.3% of wrong) █████████████
  no_boxed_anywhere               121  ( 10.7% of all,  25.1% of wrong) █████
  mcq_wrong_letter                 42  (  3.7% of all,   8.7% of wrong) █
  no_boxed_in_answer               16  (  1.4% of all,   3.3% of wrong) 
  likely_token_limit                2  (  0.2% of all,   0.4% of wrong) 
  empty_after_think                 1  (  0.1% of all,   0.2% of wrong) 

BY QUESTION TYPE

MCQ: 243/375 = 64.8%
  no_boxed_anywhere                77
  mcq_wrong_letter                 42
  no_boxed_in_answer               12
  empty_after_think                 1

Free-form: 400/751 = 53.3%
  freeform_wrong_value            301
  no_boxed_anywhere                44
  no_boxed_in_answer                4
  likely_token_limit                2

RESPONSE LENGTH DISTRI

In [15]:
# ── Show 2 examples per failure category ──────────────────────────────────────
print("=" * 60)
print("FAILURE EXAMPLES")
print("=" * 60)

INTERESTING_CATS = [
    "truncated_in_thinking",
    "no_boxed_in_answer",
    "no_boxed_anywhere",
    "empty_after_think",
    "likely_token_limit",
    "mcq_wrong_letter",
    "freeform_wrong_value",
]

for cat in INTERESTING_CATS:
    examples = by_cat_examples.get(cat, [])
    if not examples:
        continue
    print(f"\n── {cat.upper()} ({len(examples)} total) ──")
    for r in examples[:2]:
        resp = r.get("response", "")
        # Show last 500 chars (where the answer should be)
        tail = resp[-500:] if len(resp) > 500 else resp
        print(f"\n  id={r.get('id')}, is_mcq={r['is_mcq']}, gold={r.get('gold')!r}")
        print(f"  response length: {len(resp)} chars")
        print(f"  response tail:")
        for line in tail.split("\n"):
            print(f"    {line}")
        print(f"  " + "-" * 40)

FAILURE EXAMPLES

── NO_BOXED_IN_ANSWER (16 total) ──

  id=11, is_mcq=True, gold='G'
  response length: 30098 chars
  response tail:
    riangle $ FAC $
    
    We use the **base-height method**:
    
    - Base $ AC = 15 $ (given)
    - Height from $ F $ to line $ AC $: First, find the equation of line $ AC $.
    
    Slope of $ AC $: $ \frac{0 - 12}{14 - 5} = -\frac{4}{3} $
    
    Equation: $ 4x + 3y = 56 $
    
    Distance from $ F = \left( \frac{81}{8}, -\frac{27}{8} \right) $ to this line:
    
    $$
    \text{Height} = \frac{|4 \cdot \frac{81}{8} + 3 \cdot (-\frac{27}{8}) - 56|}{\sqrt{4^2 + 3^2}} = \frac{205/8}{5} = \frac{41}{8}
    $$
    
    Area of triangle $ FAC $:
    
    $$
    \text{Area
  ----------------------------------------

  id=87, is_mcq=True, gold='I'
  response length: 31019 chars
  response tail:
     Other Options
    
    Let’s quickly eliminate other options based on this:
    
    - **Option B**: Ends with `53467775360` → ends in a single zero → **

## 8. Summary

Print accuracy broken down by question type.

In [30]:
with open(SCORED_PATH) as f:
    results = [json.loads(line) for line in f]

In [31]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

In [32]:
def acc(subset):
    return sum(r.get("correct", False) for r in subset) / len(subset) * 100 if subset else 0.0

In [33]:
print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r.get('correct', False) for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r.get('correct', False) for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r.get('correct', False) for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    0 /  300  (0.00%)
  Free-form  :    0 /  643  (0.00%)
  Overall    :    0 /  943  (0.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [4]:
with open(SCORED_PATH) as fin, open(SUBMISSION_PATH, "w", newline="") as fout:
    writer = csv.DictWriter(fout, fieldnames=["id", "response"], quoting=csv.QUOTE_ALL)
    writer.writeheader()

    for line in fin:
        r = json.loads(line)
        # Keep ALL entries (including failures) — every id needs a row
        writer.writerow({
            "id": r.get("id"),
            "response": r.get("response", ""),
        })

print(f"Wrote submission to {SUBMISSION_PATH}")

Wrote submission to /content/drive/MyDrive/cse151b/submission.csv


In [7]:
with open(SUBMISSION_PATH) as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Rows: {len(rows)}")
print(f"Columns: {reader.fieldnames}")
print(f"\nFirst row id: {rows[0]['id']}")
print(f"First response (first 300 chars):")
print(rows[0]['response'][:300])
print(f"\nResponse length stats: min={min(len(r['response']) for r in rows)}, "
      f"max={max(len(r['response']) for r in rows)}, "
      f"mean={sum(len(r['response']) for r in rows)/len(rows):.0f}")

# Check no IDs are missing
ids = {r['id'] for r in rows}
print(f"\nUnique IDs: {len(ids)} (should equal row count: {len(rows)})")

# Spot-check that boxed answers survived the round-trip
n_with_boxed = sum(1 for r in rows if re.search(r'\\boxed\{', r['response']))
print(f"Rows with \\boxed{{}} present: {n_with_boxed}/{len(rows)}")

Rows: 943
Columns: ['id', 'response']

First row id: 0
First response (first 300 chars):
Okay, let's tackle these two problems one by one. I need to remember the order of operations, which is parentheses first, then exponents (though there are none here), then multiplication and division from left to right, and finally addition and subtraction from left to right. Let's start with part a

Response length stats: min=883, max=93460, mean=20029

Unique IDs: 943 (should equal row count: 943)
Rows with \boxed{} present: 783/943


In [8]:
with open(SUBMISSION_PATH) as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Total rows: {len(rows)}")
print(f"First 10 ids: {[r['id'] for r in rows[:10]]}")
print(f"Last 5 ids: {[r['id'] for r in rows[-5:]]}")

# Check ID monotonicity / completeness
ids_seen = [r['id'] for r in rows]
print(f"Unique ids: {len(set(ids_seen))}")
print(f"Any empty ids: {sum(1 for x in ids_seen if not x)}")

# Check for any rows where response looks like it leaked into id field
print(f"\nIds longer than 10 chars (suspicious): {[x for x in ids_seen if len(str(x)) > 10][:5]}")

Total rows: 943
First 10 ids: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
Last 5 ids: ['938', '939', '940', '941', '942']
Unique ids: 943
Any empty ids: 0

Ids longer than 10 chars (suspicious): []


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!